# From pixel to class: classifying Sentinel-2 near Hluboká nad Vltavou

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/image_classification_hluboka_en.ipynb)

**OpenSkyLab · notebook for the lab** [From pixel to class](https://openskylab.space/edu_viz_image_classification.html) · [česká verze](image_classification_hluboka_cs.ipynb)

In the lab you split pixels with an index threshold, compared classic classifiers and watched a neural network overfit. Here you do the same on a real Sentinel-2 image near Hluboká nad Vltavou with ESA WorldCover as reference and assess accuracy with a confusion matrix.

**Goal:** find an NDVI and SAVI threshold for vegetation, compare minimum distance, maximum likelihood, k-NN, random forest and a neural network on training and test pixels, and compute producer's and user's accuracy and kappa.

**Data:** Sentinel-2 L2A `S2B_33UVQ_20230528_0_L2A` (28 May 2023) from the Earth Search catalogue on AWS, a 4 × 4 km window around Hluboká nad Vltavou and its fishponds; reference ESA WorldCover 10 m 2021 v200. About 20 MB is downloaded.

**Tasks:** six. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 50 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Setup and data

**Required cell, run it first.** Installs the rasterio library, which reads raster data (GeoTIFF) directly from the internet, and loads the libraries, including the classifiers from scikit-learn. Defines `read_on_grid`, which reads a raster from the cloud and resamples it onto a chosen grid, and `s2_window`, which downloads the reflectance of all bands and the SCL class map of Sentinel-2 L2A from the Earth Search catalogue in a square around a point (10 m pixel). Nothing is downloaded yet. The code is collapsed; reveal it with **Show code**.

In [ ]:
# @title Setup (run first) {display-mode: "form"}
%pip install -q rasterio

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from matplotlib.colors import ListedColormap
from pyproj import Transformer
from rasterio.warp import Resampling, reproject
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

STAC = "https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/"
WORLDCOVER = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N48E012_Map.tif"
BANDS = [
    "coastal",
    "blue",
    "green",
    "red",
    "rededge1",
    "rededge2",
    "rededge3",
    "nir",
    "nir08",
    "swir16",
    "swir22",
]


def read_on_grid(url, grid, resampling=Resampling.bilinear, dtype=np.float32):
    """Read a cloud raster and resample it to grid = (transform, crs, height, width)."""
    tr, crs, ny, nx = grid
    out = np.zeros((ny, nx), dtype)
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open("/vsicurl/" + url) as src:
            reproject(
                rasterio.band(src, 1),
                out,
                dst_transform=tr,
                dst_crs=crs,
                resampling=resampling,
            )
    return out


def s2_window(item_id, lon, lat, half):
    """Reflectance of all Sentinel-2 L2A bands in a square around a point (10 m grid in the tile's UTM)."""
    item = requests.get(STAC + item_id, timeout=60).json()
    epsg = (
        item["properties"]["proj:epsg"]
        if "proj:epsg" in item["properties"]
        else int(item["properties"]["proj:code"].split(":")[1])
    )
    x, y = Transformer.from_crs(4326, epsg, always_xy=True).transform(lon, lat)
    n = int(2 * half / 10)
    grid = (
        rasterio.transform.from_origin(x - half, y + half, 10, 10),
        f"EPSG:{epsg}",
        n,
        n,
    )
    # since processing baseline 04.00 (2022) DN carry a +1000 offset; Earth Search has already removed it for newer items
    p = item["properties"]
    offset = (
        -1000
        if float(p.get("s2:processing_baseline", "0")) >= 4
        and not p.get("earthsearch:boa_offset_applied", False)
        else 0
    )
    refl = {
        b: (read_on_grid(item["assets"][b]["href"], grid) + offset) / 10000
        for b in BANDS
    }
    scl = read_on_grid(
        item["assets"]["scl"]["href"], grid, Resampling.nearest, np.uint8
    )
    wl = {b: item["assets"][b]["eo:bands"][0]["center_wavelength"] for b in BANDS}
    return refl, scl, wl, grid, item

> **💻 What the code does:** Downloads a 4 × 4 km window of a Sentinel-2 image and the ESA WorldCover 2021 map on the same grid, which serves as the reference for five classes. It takes six bands as features and keeps cloud-free pixels in these classes. It prints the number of pixels per class and plots the image and the reference.

In [ ]:
refl, scl, wl, grid, item = s2_window("S2B_33UVQ_20230528_0_L2A", 14.44, 49.05, 2000)
wc = read_on_grid(WORLDCOVER, grid, Resampling.nearest, np.uint8)
CLASSES = {10: "forest", 30: "meadow", 40: "field", 50: "built-up", 80: "water"}
COLORS = ["#2e7d32", "#9ccc65", "#f2c14e", "#c62828", "#1e88e5"]
FEAT = ["blue", "green", "red", "nir", "swir16", "swir22"]
X_img = np.dstack([refl[b] for b in FEAT])
ok = np.isin(scl, [4, 5, 6]) & np.isin(wc, list(CLASSES))
print(
    "pixels per WorldCover class:",
    {CLASSES[k]: int(((wc == k) & ok).sum()) for k in CLASSES},
)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(
    np.clip(np.dstack([refl["red"], refl["green"], refl["blue"]]) / 0.15, 0, 1)
)
ax[0].set_title("Sentinel-2, 28 May 2023")
lab = np.full(wc.shape, np.nan)
[lab.__setitem__(wc == k, i) for i, k in enumerate(CLASSES)]
ax[1].imshow(lab, cmap=ListedColormap(COLORS), vmin=0, vmax=4, interpolation="nearest")
ax[1].set_title("ESA WorldCover 2021 (reference)")
for a in ax:
    a.axis("off")
plt.show()

## 2. NDVI threshold: two errors against each other

Vegetation = forest, meadow and field according to WorldCover, the rest = built-up and water. For each threshold we count how much vegetation we miss and how much non-vegetation we label as vegetation.

> **💻 What the code does:** Computes NDVI and SAVI with the soil factor `L`. For thresholds from −0.2 to 0.9 it computes how much WorldCover vegetation the threshold would miss, how many other pixels it would label as vegetation, and the overall accuracy, and plots this for both indices with the best threshold.

In [ ]:
# 0 = NDVI, 0.5 usual, 1 sparse vegetation.
L = 0.5  # ◀ parameter: SAVI soil adjustment factor
nir, red = refl["nir"], refl["red"]
ndvi = (nir - red) / (nir + red)
savi = (1 + L) * (nir - red) / (nir + red + L)
veg = np.isin(wc, [10, 30, 40])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, (name, idx) in zip(ax, [("NDVI", ndvi), (f"SAVI (L = {L})", savi)]):
    thr = np.linspace(-0.2, 0.9, 111)
    miss = [np.mean(idx[ok & veg] < t_) for t_ in thr]  # vegetation missed
    # Built-up and water as vegetation.
    false = [np.mean(idx[ok & ~veg] >= t_) for t_ in thr]
    total = [np.mean((idx[ok] >= t_) == veg[ok]) for t_ in thr]
    best = thr[int(np.argmax(total))]
    a.plot(thr, miss, label="vegetation missed")
    a.plot(thr, false, label="other as vegetation")
    a.plot(thr, total, "k", label="overall accuracy")
    a.axvline(best, color="gray", ls=":")
    a.set(
        xlabel=name,
        title=f"{name}: best threshold {best:.2f}, accuracy {max(total):.1%}",
    )
ax[0].legend()
plt.show()

> **✏️ Task 1.** Which NDVI threshold is best and which errors remain? Why can you not get rid of both errors at once? Which “vegetation” pixels have a low NDVI (look at the fields: after harvest there is stubble, in spring the soil may still be bare) and why are they counted as vegetation in WorldCover?

*Your answer:* 

> **✏️ Task 2.** Compare NDVI and SAVI. Set `L = 0`, `0.25` and `1`. How do the range of values and the best threshold change? Why does SAVI depend less on soil brightness and why does it hardly matter in dense vegetation?

*Your answer:* 

## 3. Training and test pixels

Neighbouring pixels are similar, so a random split would overstate accuracy. We therefore train on the northern half of the window and test on the southern half.

We take **up to `n_train` pixels of each class** in the north for training and up to
2,000 pixels of each class in the south for testing. If fewer suitable pixels exist,
we take all of them. The mask `ok` already restricts the pixels to a suitable SCL class
and to one of the selected WorldCover reference classes.

North and south are two copies of the mask `ok`: in the northern mask we switch off
the bottom half of the rows, in the southern one the top half. The function `sample` then, for each class:

1. Uses the mask to select its pixels in the given half of the image.
2. Builds `class_features`: each row is one pixel, the columns are its
   six spectral features.
3. Picks rows at random with `rng.choice`. `replace=False` means the same
   pixel is never picked twice. `min` limits the requested count to the available pixels.
4. Appends the reference class converted to a number 0–4 in the order of `CLASSES`.

The output `X` is the feature table of the selected pixels and `y` their reference classes.
`np.vstack` stacks the groups of pixels on top of each other. Sampling per class limits
the dominance of abundant classes; the test accuracy therefore does not reflect the natural
areal share of the classes in the whole landscape.

The seed `1` makes the whole selection repeatable. Each call of `sample` advances
the state of the generator. **After changing `n_train`, run this cell and the following
models again.**

In [ ]:
n_train = 300  # ◀ parameter: at most this many training pixels per class
rng = np.random.default_rng(1)

middle_row = wc.shape[0] // 2
north = ok.copy()
north[middle_row:, :] = False
south = ok.copy()
south[:middle_row, :] = False


def sample(region_mask, n):
    """Select up to n pixels of each class in the given region, without repetition."""
    feature_groups, labels = [], []
    for class_index, class_code in enumerate(CLASSES):
        class_mask = region_mask & (wc == class_code)
        class_features = X_img[class_mask]
        sample_count = min(n, len(class_features))

        chosen_features = rng.choice(class_features, size=sample_count, replace=False)
        feature_groups.append(chosen_features)
        labels.extend([class_index] * sample_count)

    return np.vstack(feature_groups), np.array(labels)


Xtr, ytr = sample(north, n_train)
Xte, yte = sample(south, 2000)
print("training:", np.bincount(ytr), "test:", np.bincount(yte))

### How we will evaluate the classifiers

The function `report` compares the model's prediction on the test pixels with the WorldCover
reference classes. It returns the confusion matrix with producer's (PA) and
user's (UA) accuracy and prints the overall accuracy on the test and training pixels and the kappa coefficient.
We use the same function for all models below.

In [ ]:
names = list(CLASSES.values())


def report(model, title):
    pred = model.predict(Xte)
    cm = confusion_matrix(yte, pred, labels=range(5))
    pa, ua = np.diag(cm) / cm.sum(1), np.diag(cm) / np.maximum(cm.sum(0), 1)
    tab = pd.DataFrame(
        cm.astype(object), index=[f"ref {n}" for n in names], columns=names
    )
    tab["PA"] = pa.round(2)
    tab.loc["UA"] = list(ua.round(2)) + [""]
    print(
        f"{title}: overall accuracy test {np.mean(pred == yte):.1%}, training {model.score(Xtr, ytr):.1%}, kappa {cohen_kappa_score(yte, pred):.2f}"
    )
    return tab

## 4. Minimum distance and maximum likelihood

> **💻 What the code does:** Trains a minimum-distance classifier and Gaussian maximum likelihood (QDA) and shows the confusion matrix on the test pixels for both.

In [ ]:
mind = NearestCentroid().fit(Xtr, ytr)
display(report(mind, "Minimum distance"))
# Gaussian maximum likelihood.
ml = QuadraticDiscriminantAnalysis(reg_param=1e-3).fit(Xtr, ytr)
display(report(ml, "Maximum likelihood"))

> **✏️ Task 3.** Read the maximum likelihood confusion matrix: rows are the reference, columns the result. Which class has a low producer's accuracy (PA, it is missed) and which a low user's accuracy (UA, other classes fall into it)? Which pairs of classes are confused most and why? Which of the two methods did better? Maximum likelihood accounts for the spread and correlation of bands within each class; why might it still not win (classes are not normally distributed, meadow and forest have similar spectra, the reference has errors)?

*Your answer:* 

## 5. How complex a model? k-NN and random forest

> **💻 What the code does:** Trains k-NN for 1 to 100 neighbours and a random forest of 200 trees for depths from 1 to unlimited and plots the accuracy on the training and test pixels.

In [ ]:
ks = [1, 3, 5, 10, 25, 50, 100]
depths = [1, 2, 3, 5, 8, 12, None]
acc = {"k-NN training": [], "k-NN test": [], "forest training": [], "forest test": []}
for k_ in ks:
    m = KNeighborsClassifier(k_).fit(Xtr, ytr)
    acc["k-NN training"].append(m.score(Xtr, ytr))
    acc["k-NN test"].append(m.score(Xte, yte))
for d in depths:
    m = RandomForestClassifier(200, max_depth=d, random_state=0, n_jobs=-1).fit(
        Xtr, ytr
    )
    acc["forest training"].append(m.score(Xtr, ytr))
    acc["forest test"].append(m.score(Xte, yte))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(ks, acc["k-NN training"], "o-", label="training")
ax[0].plot(ks, acc["k-NN test"], "s-", label="test")
ax[0].set(
    xscale="log",
    xlabel="k (number of neighbours)",
    ylabel="overall accuracy",
    title="k-NN",
)
ax[0].legend()
lbl = [str(d) for d in depths]
ax[1].plot(lbl, acc["forest training"], "o-", label="training")
ax[1].plot(lbl, acc["forest test"], "s-", label="test")
ax[1].set(xlabel="max. tree depth", title="random forest (200 trees)")
ax[1].legend()
plt.show()

> **✏️ Task 4.** With k = 1 the accuracy on training pixels is 100 % or nearly so. Why does that mean nothing? Which k and which tree depth give the best test accuracy? Describe what overfitting and underfitting look like in both plots.

*Your answer:* 

## 6. A neural network and overfitting

A small network with one hidden layer. We train it on few pixels and watch training and test accuracy over the epochs.

> **💻 What the code does:** On a small sample of up to `n_small` pixels per class it trains a neural network with one hidden layer of `hidden` neurons for 400 epochs and records the training and test accuracy after each epoch. It plots both curves and prints the final and best test accuracy. `sample(north, n_small)` draws a new selection from the northern half; it need not be a subset of `Xtr`. Running this cell again advances the generator and selects different pixels; to repeat the same run, start again from the cell that selects the training and test pixels.

In [ ]:
hidden = 64  # ◀ parameter: number of hidden neurons
# A small training sample overfits more easily.
n_small = 30  # ◀ parameter: at most this many training pixels per class
Xs, ys = sample(north, n_small)
mu, sd = Xs.mean(0), Xs.std(0)
net = MLPClassifier((hidden,), learning_rate_init=0.01, alpha=0.0, random_state=0)
hist = []
for epoch in range(400):
    net.partial_fit((Xs - mu) / sd, ys, classes=range(5))
    hist.append((net.score((Xs - mu) / sd, ys), net.score((Xte - mu) / sd, yte)))
hist = np.array(hist)
plt.figure(figsize=(8, 3.5))
plt.plot(hist[:, 0], label="training")
plt.plot(hist[:, 1], label="test")
plt.xlabel("epoch")
plt.ylabel("overall accuracy")
plt.title(f"network with {hidden} neurons, {n_small} pixels per class")
plt.legend()
plt.show()
print(
    f"end: training {hist[-1, 0]:.1%}, test {hist[-1, 1]:.1%}; best test {hist[:, 1].max():.1%} at epoch {hist[:, 1].argmax()}"
)

> **✏️ Task 5.** Try `hidden` 4, 64 and 256 and `n_small` 30 and 300. When does the network overfit (training rises, test falls or stalls)? What would help: more data, a smaller network or stopping earlier? Why is the training–test gap larger for the network than for maximum likelihood?

*Your answer:* 

## 7. Result map

For the final map we use `forest`: a random forest with **200 trees and a maximum
depth of 12**. Its settings are fixed here in advance; the code does not automatically pick
the best model from the comparison above. After training we show its confusion
matrix with the same `report` function.

The classifier expects one pixel per row, so we turn the input image into a
feature table and the prediction `pred` back into a map. Converting to `float`
lets us mark hidden pixels with `NaN`.

The mask `visible_pixels` keeps the SCL classes **4 = vegetation, 5 = not vegetated
surfaces, 6 = water**. `~visible_pixels` selects all other pixels, for example
clouds, shadows or snow; setting them to `NaN` hides them in the map. This is a display
mask: unlike the training mask `ok`, it does not require a selected WorldCover
class.

The WorldCover 2021 reference stays on the left, the prediction from Sentinel-2 2023 on the right.
The white line in the right map marks the split into the training north and the test south.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=200, max_depth=12, random_state=0, n_jobs=-1
)
forest.fit(Xtr, ytr)
display(report(forest, "Random forest, depth 12"))
pred = forest.predict(X_img.reshape(-1, len(FEAT))).reshape(wc.shape).astype(float)
# Vegetation (4), not vegetated surfaces (5) and water (6).
visible_pixels = np.isin(scl, [4, 5, 6])
# Hide all other pixels in the map.
pred[~visible_pixels] = np.nan
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
maps = [(lab, "WorldCover 2021"), (pred, "random forest, Sentinel-2 2023")]
for axis, (img, title) in zip(axes, maps):
    axis.imshow(
        img, cmap=ListedColormap(COLORS), vmin=0, vmax=4, interpolation="nearest"
    )
    axis.set_title(title)
    axis.axis("off")
axes[1].axhline(wc.shape[0] / 2, color="w", lw=1)
plt.show()

> **✏️ Task 6.** Find places where the map differs from WorldCover. Is the error always in the classification? Consider that WorldCover is from 2021 and has errors of its own, that a single image from one day cannot tell a meadow from a field of forage crops, nor green spring cereals from grass, and that a 10 m pixel at a forest edge is a mixture. How would you assess accuracy honestly?

*Your answer:* 

> **✏️ Extension (optional).** Compare your results with a classmate who has the notebook for another place (Roztoky, Brno, Hluboká nad Vltavou). Which classes are confused more elsewhere, and why (season of the image, type of landscape, field size)?

*Your answer:* 

---
### Data sources and references

- Copernicus Sentinel-2 data 2023, ESA (L2A, Sen2Cor); AWS Open Data registry (`sentinel-cogs`), Earth Search catalogue (Element 84). Free use with attribution.
- Zanaga, D. et al. (2022). *ESA WorldCover 10 m 2021 v200*. https://doi.org/10.5281/zenodo.7254221 (CC BY 4.0).
- Huete, A. R. (1988). A soil-adjusted vegetation index (SAVI). *Remote Sensing of Environment* 25, 295–309. https://doi.org/10.1016/0034-4257(88)90106-X
- Congalton, R. G., Green, K. (2019). *Assessing the Accuracy of Remotely Sensed Data: Principles and Practices*, 3rd ed. CRC Press.
- Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research* 12, 2825–2830.

### How to cite

OpenSkyLab (2026). From pixel to class: classifying Sentinel-2 near Hluboká nad Vltavou [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/image_classification_hluboka_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*